# 1. Data preprocessing

**NLP Final Term Project, Group 02 — detecting machine-generated text.**

This notebook turns two raw corpora into the one fixed, balanced, leakage-checked
split that every later notebook reuses. Nothing here trains a model.

| | source | task |
|---|---|---|
| D1 | DAIGT V2 (`daigt.csv`) | student essays, human against machine |
| D2 | HC3 (`hc3.jsonl`) | question answering, human against ChatGPT |

Steps, in order:

1. load each corpus and reduce it to `text` and a binary `label` (0 human, 1 machine),
2. balance the classes by downsampling the majority class,
3. hash the normalised text so near-identical documents form groups,
4. split 72 / 8 / 20 into train / validation / test **without letting a duplicate group
   cross a partition boundary**,
5. assert that no group crossed, and record the label balance of each partition,
6. build the two cleaning paths the models need: a classical lemmatised bag-of-words
   path, and the subword tokenisation the transformers consume.

Writes `experiments/paper_scale/work/data_{tag}.parquet` and `split_{tag}.npz`.
Notebooks 02, 03 and 04 read those files and never rebuild them, so the evaluation
set is identical across every result in the report.

## 1.1 Environment and paths

In [1]:
import os

os.environ.setdefault('HF_HOME', '/media/filwel/MLProject1/hf_cache')
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS', '1')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import gc
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

# Raw corpora live outside the repository; the repository holds the derived splits.
PROJECT_DIR = Path('/media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Project ')

FINAL_DIR = Path('/media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final')
if not FINAL_DIR.exists():
    p = Path.cwd().resolve()
    while p.name != 'Final' and p != p.parent:
        p = p.parent
    FINAL_DIR = p

PS_DIR = FINAL_DIR / 'experiments' / 'paper_scale'
WORK_DIR = PS_DIR / 'work'
RESULTS_DIR = PS_DIR / 'results'
PROBS_DIR = PS_DIR / 'probs'
MODELS_DIR = PS_DIR / 'models'
CKPT_DIR = Path('/media/filwel/MLProject1/nlp_paper_ckpt')

MAX_LEN = 128
EPOCHS = 5
WARMUP_RATIO = 0.1
PATIENCE = 2
SPLIT_SEED = 42
TRAIN_SEED = 42

MODELS = {'BERT': 'bert-base-uncased', 'DeBERTa': 'microsoft/deberta-v3-base'}
DATASET_NAMES = {'D1': 'DAIGT V2', 'D2': 'HC3'}

WORK_DIR.mkdir(parents=True, exist_ok=True)
print('work directory:', WORK_DIR)

work directory: /media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final/experiments/paper_scale/work


## 1.2 Loading and class balancing

Both corpora arrive unbalanced. HC3 additionally arrives one row per *question*, with
a list of human answers and a list of ChatGPT answers in that row, so it is exploded
to one answer per row before anything else happens.

Balancing is a downsample of the majority class to the minority count, at a fixed
seed. Accuracy on a balanced test set is then directly interpretable: 0.5 is chance.

In [2]:
def normalise(t):
    """Collapse runs of whitespace. The only text change applied before hashing,
    splitting, or transformer tokenisation."""
    return re.sub(r'\s+', ' ', str(t)).strip()


def balance(df, seed=SPLIT_SEED):
    n = int(df['label'].value_counts().min())
    parts = [df[df['label'] == v].sample(n=n, random_state=seed)
             for v in sorted(df['label'].unique())]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)


def load_D1():
    """DAIGT V2: already one document per row, with a 0/1 label column."""
    raw = pd.read_csv(PROJECT_DIR / 'daigt.csv')
    df = raw[['text', 'label']].dropna()
    df['label'] = df['label'].astype(int)
    del raw
    gc.collect()
    return balance(df)


def load_D2():
    """HC3: one row per question holding two answer lists. Explode both, label,
    and concatenate."""
    raw = pd.read_json(PROJECT_DIR / 'hc3.jsonl', lines=True)
    human = raw[['human_answers']].explode('human_answers').rename(
        columns={'human_answers': 'text'})
    human['label'] = 0
    bot = raw[['chatgpt_answers']].explode('chatgpt_answers').rename(
        columns={'chatgpt_answers': 'text'})
    bot['label'] = 1
    df = pd.concat([human, bot], ignore_index=True).dropna()
    df['text'] = df['text'].astype(str)
    del raw, human, bot
    gc.collect()
    return balance(df)


LOADERS = {'D1': load_D1, 'D2': load_D2}

## 1.3 Duplicate-group-aware split

HC3 contains a substantial number of near-identical answers: 7.16 percent of the
corpus is a repeat of some other row once whitespace and case are normalised
(measured in `experiments/audit/hc3_full_audit.json`). A plain stratified split puts
copies of the same answer on both sides of the train/test boundary, and the test
score then partly measures memorisation.

So rows are grouped by the MD5 of their normalised lowercased text, and
`GroupShuffleSplit` keeps a whole group on one side. Measured effect on HC3: the
group-aware split leaks **0 of 10,732** test rows, the naive split leaks **570 of
10,762**, which is 5.30 percent. DAIGT's duplication rate is 0.01 percent, so
grouping is very nearly a no-op there, but it is applied to both datasets for
consistency.

The shares are 72 / 8 / 20, not 80 / 10 / 10: the first split takes a fifth for
test, the second takes a tenth of the remaining 80 percent for validation.

In [3]:
import hashlib

from sklearn.model_selection import GroupShuffleSplit


def content_hash(series):
    return series.map(lambda t: hashlib.md5(normalise(t).lower().encode()).hexdigest())


def group_split(df, seed=SPLIT_SEED):
    groups = df['hash'].values
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_full, te = next(gss1.split(df, df['label'], groups))
    sub = df.iloc[tr_full]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=seed)
    tr_rel, val_rel = next(gss2.split(sub, sub['label'], sub['hash'].values))
    idx_tr, idx_val = sub.index.values[tr_rel], sub.index.values[val_rel]
    idx_te = df.index.values[te]
    g_tr = set(df.loc[idx_tr, 'hash'])
    g_val = set(df.loc[idx_val, 'hash'])
    g_te = set(df.loc[idx_te, 'hash'])
    assert not (g_tr & g_val) and not (g_tr & g_te) and not (g_val & g_te), \
        'GROUP LEAKAGE ACROSS SPLIT'
    return idx_tr, idx_val, idx_te


def build_or_load_splits(tag, rebuild=False):
    """Build the split once and cache it. Every later notebook loads this file."""
    data_p = WORK_DIR / f'data_{tag}.parquet'
    split_p = WORK_DIR / f'split_{tag}.npz'
    if not rebuild and data_p.exists() and split_p.exists():
        df = pd.read_parquet(data_p)
        sp = np.load(split_p)
        print(f'{tag}: loaded cached split from disk')
        return df, {'train': sp['train'], 'val': sp['val'], 'test': sp['test']}
    df = LOADERS[tag]()
    df['hash'] = content_hash(df['text'])
    n_groups = df['hash'].nunique()
    print(f'{tag}: balanced rows={len(df)}  unique content groups={n_groups}  '
          f'duplicate rows={len(df) - n_groups} '
          f'({100 * (len(df) - n_groups) / len(df):.2f} percent)')
    idx_tr, idx_val, idx_te = group_split(df)
    df[['text', 'label']].to_parquet(data_p, index=True)
    np.savez(split_p, train=idx_tr, val=idx_val, test=idx_te)
    print(f'{tag}: written {data_p.name} and {split_p.name}')
    return df[['text', 'label']], {'train': idx_tr, 'val': idx_val, 'test': idx_te}


DATA, SPLITS = {}, {}
for tag in ('D1', 'D2'):
    DATA[tag], SPLITS[tag] = build_or_load_splits(tag)

D1: loaded cached split from disk


D2: loaded cached split from disk


## 1.4 Split integrity

Three things are checked, because each of them would quietly inflate the reported
scores if it were wrong: partition sizes, class balance inside every partition, and
whether any normalised document appears in more than one partition.

In [4]:
rows = []
for tag in ('D1', 'D2'):
    df, sp = DATA[tag], SPLITS[tag]
    seen = {}
    for split in ('train', 'val', 'test'):
        sub = df.loc[sp[split]]
        counts = sub['label'].value_counts().to_dict()
        rows.append({'dataset': f'{tag} {DATASET_NAMES[tag]}', 'split': split,
                     'n': len(sub), 'n_human': counts.get(0, 0), 'n_ai': counts.get(1, 0),
                     'ai_fraction': round(counts.get(1, 0) / len(sub), 4)})
        seen[split] = set(content_hash(sub['text']))
    overlap = (len(seen['train'] & seen['test']), len(seen['train'] & seen['val']),
               len(seen['val'] & seen['test']))
    print(f'{tag} {DATASET_NAMES[tag]:9s} shared normalised documents '
          f'train-test={overlap[0]}  train-val={overlap[1]}  val-test={overlap[2]}')
    assert sum(overlap) == 0, f'{tag}: content leaked across partitions'

print()
print(pd.DataFrame(rows).to_string(index=False))

D1 DAIGT V2  shared normalised documents train-test=0  train-val=0  val-test=0


D2 HC3       shared normalised documents train-test=0  train-val=0  val-test=0

    dataset split     n  n_human  n_ai  ai_fraction
D1 DAIGT V2 train 25196    12549 12647       0.5019
D1 DAIGT V2   val  2800     1429  1371       0.4896
D1 DAIGT V2  test  6998     3519  3479       0.4971
     D2 HC3 train 38785    19388 19397       0.5001
     D2 HC3   val  4289     2138  2151       0.5015
     D2 HC3  test 10732     5377  5355       0.4990


## 1.5 Classical cleaning path

The classical baselines (Naive Bayes, logistic regression, linear SVM over
bag-of-words and TF-IDF) need a heavier normalisation than the transformers do:
lowercase, strip everything that is not a letter, drop English stopwords and
one-character tokens, and lemmatise.

This path is **not** applied before the transformers. Casing, punctuation and
function words carry signal that a subword model can use, and discarding them
before fine-tuning would throw away part of what the model is being asked to
detect.

In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

for pkg in ('punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4'):
    nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))


def preprocess_classical(text):
    text = re.sub(r'[^a-z\s]', ' ', str(text).lower())
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)


demo = DATA['D2']['text'].iloc[0]
print('raw       ', repr(demo[:150]))
print()
print('normalised', repr(normalise(demo)[:150]))
print()
print('classical ', repr(preprocess_classical(demo)[:150]))

raw        'Bond ETFs can drop in value when the dividends are paid out, but the magnitude of the drop will depend on the specific ETF and the prevailing market c'

normalised 'Bond ETFs can drop in value when the dividends are paid out, but the magnitude of the drop will depend on the specific ETF and the prevailing market c'



classical  'bond etf drop value dividend paid magnitude drop depend specific etf prevailing market condition important note bond etf like etf subject market fluct'


## 1.6 Transformer tokenisation

Each transformer uses its own subword vocabulary, so tokenisation happens per model:
WordPiece for BERT, SentencePiece for DeBERTa-v3. Sequences are truncated at 128
tokens and padded per batch rather than to a global maximum, which keeps the padded
fraction low.

128 is a project constraint carried over from the midterm, not a free choice. The
cell below reports what it costs: the share of documents that reach the limit and
are therefore cut.

In [6]:
from datasets import Dataset
from transformers import AutoTokenizer

_TOKCACHE = {}


def get_tokenizer(model_key):
    if model_key not in _TOKCACHE:
        _TOKCACHE[model_key] = AutoTokenizer.from_pretrained(MODELS[model_key])
    return _TOKCACHE[model_key]


def tokenise_split(tag, model_key, split):
    """Tokenise one partition into the exact form the Trainer consumes."""
    tok = get_tokenizer(model_key)
    sub = DATA[tag].loc[SPLITS[tag][split]]
    ds = Dataset.from_dict({'text': [normalise(t) for t in sub['text']],
                            'labels': [int(v) for v in sub['label']]})
    return ds.map(lambda b: tok(b['text'], truncation=True, max_length=MAX_LEN),
                  batched=True, remove_columns=['text'])


example = tokenise_split('D1', 'BERT', 'val')
print(example)
print()
print('first row input_ids (truncated view):', example[0]['input_ids'][:24], '...')
print('decoded:', get_tokenizer('BERT').decode(example[0]['input_ids'][:24]))

Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

Map:  36%|███▌      | 1000/2800 [00:00<00:01, 1685.70 examples/s]

Map:  71%|███████▏  | 2000/2800 [00:01<00:00, 1731.54 examples/s]

Map: 100%|██████████| 2800/2800 [00:01<00:00, 1744.10 examples/s]

Map: 100%|██████████| 2800/2800 [00:01<00:00, 1732.84 examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2800
})

first row input_ids (truncated view): [101, 1057, 5603, 1010, 1045, 2123, 1005, 1056, 10587, 4339, 2178, 9491, 1010, 2021, 1045, 3984, 2009, 1005, 1055, 2005, 1037, 2204, 3426, 1012] ...
decoded: [CLS] ugh, i don ' t wanna write another essay, but i guess it ' s for a good cause.


In [7]:
# Length diagnostic on a fixed 4000-row sample per dataset per tokenizer.
SAMPLE_N = 4000
rows = []
for tag in ('D1', 'D2'):
    sub = DATA[tag].loc[SPLITS[tag]['train']]
    sub = sub.sample(n=min(SAMPLE_N, len(sub)), random_state=SPLIT_SEED)
    texts = [normalise(t) for t in sub['text']]
    for model_key in ('BERT', 'DeBERTa'):
        tok = get_tokenizer(model_key)
        lens = np.array([len(x) for x in
                         tok(texts, add_special_tokens=True, truncation=False)['input_ids']])
        rows.append({'dataset': f'{tag} {DATASET_NAMES[tag]}', 'tokenizer': model_key,
                     'median': int(np.median(lens)),
                     'p90': int(np.percentile(lens, 90)),
                     'p99': int(np.percentile(lens, 99)),
                     'max': int(lens.max()),
                     f'pct_over_{MAX_LEN}': round(100 * float((lens > MAX_LEN).mean()), 2)})

print(pd.DataFrame(rows).to_string(index=False))

Token indices sequence length is longer than the specified maximum sequence length for this model (606 > 512). Running this sequence through the model will result in indexing errors


    dataset tokenizer  median  p90  p99  max  pct_over_128
D1 DAIGT V2      BERT     414  688 1088 1626         99.62
D1 DAIGT V2   DeBERTa     408  671 1070 1605         99.52
     D2 HC3      BERT     163  307  713 1553         61.10
     D2 HC3   DeBERTa     158  299  685 1468         59.82


## 1.7 What this notebook produced

`data_D1.parquet`, `split_D1.npz`, `data_D2.parquet`, `split_D2.npz` in
`experiments/paper_scale/work/`.

Notebook 02 (BERT), notebook 03 (DeBERTa) and notebook 04 (ensemble) all read these
same four files. Because the split is fixed and shared, the two models are scored on
byte-identical test rows, which is what makes the paired ensemble comparison in
notebook 04 legitimate.

In [8]:
for tag in ('D1', 'D2'):
    for name in (f'data_{tag}.parquet', f'split_{tag}.npz'):
        p = WORK_DIR / name
        print(f'{name:24s} {p.stat().st_size / 1024 ** 2:8.2f} MB   {p}')

data_D1.parquet             38.22 MB   /media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final/experiments/paper_scale/work/data_D1.parquet
split_D1.npz                 0.27 MB   /media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final/experiments/paper_scale/work/split_D1.npz
data_D2.parquet             25.02 MB   /media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final/experiments/paper_scale/work/data_D2.parquet
split_D2.npz                 0.41 MB   /media/filwel/All/Sakib/Semester 10/ NATURAL LANGUAGE PROCESSING /Final/experiments/paper_scale/work/split_D2.npz
